In [ ]:
!pip install -q pypdf faiss-cpu sentence-transformers transformers accelerate

In [ ]:
from google.colab import files
from pypdf import PdfReader
from tqdm import tqdm

uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]

reader = PdfReader(pdf_name)

pages = []
page_no = 1
for page in tqdm(reader.pages, desc="PDF parh raha hoon"):
    text = page.extract_text()
    if text:
        text = text.replace("\n", " ")        # naye lines hata do
        text = " ".join(text.split())         # faltu spaces hata do
        pages.append({"page": page_no, "text": text})
    page_no = page_no + 1

print("Text wale pages:", len(pages))

Saving n8n_automation_portfolio.pdf to n8n_automation_portfolio.pdf


PDF parh raha hoon: 100%|██████████| 2/2 [00:00<00:00, 141.14it/s]

Text wale pages: 2


In [ ]:
chunks = []

for p in pages:
    text = p["text"]
    start = 0
    while start < len(text):
        piece = text[start : start + 800]
        if len(piece) > 50:                   # bohat chhote tukre chhor do
            chunks.append({"text": piece, "page": p["page"]})
        start = start + 650                   # 150 overlap

print("Total tukre:", len(chunks))

Total tukre: 4


In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

all_texts = [c["text"] for c in chunks]
vectors = embedder.encode(all_texts, normalize_embeddings=True, show_progress_bar=True)
vectors = np.array(vectors).astype("float32")

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print("Taiyar! Index mein tukre:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Taiyar! Index mein tukre: 4


In [ ]:
import torch
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

# Reranker: tukron ko sawal ke hisaab se dobara number deta hai
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Naya AI model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if device == "cuda" else torch.float32
)
model = model.to(device)

print("Model load ho gaya!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

... (9 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model load ho gaya!


In [ ]:
# Function 1: AI se jawab lo
def generate_text(system_text, user_text, max_tokens=300):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    output = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]     # sirf naya likha hua hissa
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Function 2: sawal ka jawab dhoondo
def ask(question):
    # Step 1: FAISS se 10 milte julte tukre
    q_vector = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_vector, 10)

    # agar sab se achha score bhi kam hai, to jawab PDF mein nahi hai
    if scores[0][0] < 0.2:
        return "Mujhe document mein is sawal ka jawab nahi mila.", []

    candidates = [chunks[i] for i in ids[0] if i != -1]

    # Step 2: Reranker in 10 mein se sab se achhe 4 chunta hai
    pairs = [[question, c["text"]] for c in candidates]
    rerank_scores = reranker.predict(pairs)
    order = np.argsort(rerank_scores)[::-1][:4]
    best = [candidates[i] for i in order]

    # Step 3: AI ko tukre dekar jawab likhwao
    context = ""
    number = 1
    for c in best:
        context = context + "[Passage " + str(number) + ", Page " + str(c["page"]) + "] " + c["text"] + "\n\n"
        number = number + 1

    system_text = ("You answer questions using ONLY the given passages. "
                   "Give a clear, complete answer in 2 to 5 sentences. "
                   "If the passages do not contain the answer, reply exactly: "
                   "I could not find the answer in the document. "
                   "Never make up information.")
    user_text = "Passages:\n" + context + "Question: " + question

    answer = generate_text(system_text, user_text)
    return answer, best

# Function 3: poore document ka khulasa
def summarize():
    step = max(1, len(chunks) // 8)
    text = ""
    for c in chunks[::step][:8]:
        text = text + c["text"] + "\n"
    return generate_text("You write short clear summaries.",
                         "Summarize this document in 5 sentences:\n" + text)

print("Sab kuch taiyar hai!")

Sab kuch taiyar hai!


In [ ]:
answer, sources = ask("Which embedding model should be used?")
print("Jawab:", answer)
print()
for s in sources:
    print("Page", s["page"], ":", s["text"][:150])
    print("---")

Jawab: Mujhe document mein is sawal ka jawab nahi mila.



In [ ]:
chat_history = []

print("PDF se sawal poochho (English mein behtar chalta hai)!")
print("Commands:  summary  |  clear  |  save  |  exit\n")

while True:
    question = input("Aap: ")

    if question.lower() == "exit":
        print("Allah hafiz!")
        break

    elif question.lower() == "clear":
        chat_history = []
        print("Chat saaf kar di gayi.\n")

    elif question.lower() == "summary":
        print("Khulasa banaya ja raha hai...")
        result = summarize()
        print("Bot:", result, "\n")
        chat_history.append("Aap: summary\nBot: " + result)

    elif question.lower() == "save":
        f = open("chat.txt", "w")
        f.write("\n\n".join(chat_history))
        f.close()
        files.download("chat.txt")

    elif question.strip() == "":
        print("Pehle kuch likho.\n")

    else:
        answer, sources = ask(question)
        print("Bot:", answer)

        if len(sources) > 0:
            print("\nSources:")
            for s in sources:
                print("  Page", s["page"], ":", s["text"][:100], "...")
        print()

        chat_history.append("Aap: " + question + "\nBot: " + answer)

PDF se sawal poochho (English mein behtar chalta hai)!
Commands:  summary  |  clear  |  save  |  exit

Aap: clear
Chat saaf kar di gayi.

Aap: exit
Allah hafiz!
